 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [ ]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [ ]:
PROMPT_END_STR_PERPLEX = '---'    
RESPONSE_SOURCES_DIVIDER_STR = '<div style="text-align: center">⁂</div>'
source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')

perplex_source_list_OLD_FORMATre = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
# sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)') # note used?

relinker = lpz.relinker

def split_single_prompt_response_text_perplex(pr_text: str) -> Tuple[str, str, str, str]:
    """Splits perplexity output markdown text into prompt, response and source sections.
    In the response, duplicate citenums are removed, and the mapping from original 
    to deduplicated numbers is in citenumes_to_url_source"""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', pr_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    preamble = pr_text[:heading_start_index].strip()
    
    prompt_end_index, response_start_index = rfw.find_markdown_divider_boundaries(pr_text)
        
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}. '
                         'Probably missed the starting level 1 header part of the prompt.')
    
    prompt = pr_text[heading_start_index:prompt_end_index+1].strip()

    response_sources_divider_index = pr_text.rfind(RESPONSE_SOURCES_DIVIDER_STR)

    if response_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if response_sources_divider_index <= response_start_index:
        raise ValueError('body_sources_divider_index <= response_sources_divider_index')
    
    response = f"{pr_text[response_start_index:response_sources_divider_index]}".strip()
    
    sources = pr_text[response_sources_divider_index:]
    citenum_url_pairs = rfw.get_link_tu_pairs(sources, source_list_pattern_perplex)
    
    return lpz.PromptResponseSplit(preamble, prompt, response, citenum_url_pairs, None) # no source titles

In [ ]:
# def split_single_prompt_response_dedup_perplex(markdown_text: str) -> lpz.PromptResponseSplitDeDup:
#     """Splits perplexity output markdown text into prompt, response and source sections.
#     In the response, duplicate citenums are removed, and the mapping from original 
#     to deduplicated numbers is in citenumes_to_url_source"""
#     raise ValueError('should not be calling this anymore')
#     return relinker.split_prompt_response_dedup(markdown_text, split_single_prompt_response_text_perplex)

def relink_single_file_perplexity(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    "Relinks and writes to a file a single prompt/response from perplexity."
    file_text = lpz.read_markdown_file(perplexity_file)
    prsplit = relinker.split_prompt_response_dedup(file_text, split_single_prompt_response_text_perplex)
    #prsplit = split_single_prompt_response_dedup_perplex(file_text)
    
    body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit,'plain_link')
    body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)

    relinked_file.write_text(f'{lpz.make_obsidian_front_matter()}\n*{source_link}*\n# Prompt\n\n{prsplit.prompt}\n'
                             f'# Response\n\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', 
                             encoding='utf-8')

In [ ]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
# perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / "perple_new_format_longprompt_example.md"
perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex' / 'GPT-4o.md'

output_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Scratch Space")

output_file = output_dir / "tmp_perplex_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_single_file_perplexity(perplexity_dialog_file, output_file, verbose)
print('Done.')

### Test merging

In [ ]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

# multi-file, same prompt
#chat_files = list(datdir.glob('*.md'))
# multi-file, different prompt
#chat_files = [chat_files[3], pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_example.md")]
# single file
#chat_files = [chat_files[3]]

# single smc file but multiprompt
chat_files = [pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_multi_prompt_savemychatbot_example.md")]


merged_output_file = output_dir / 'tmp_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [ ]:
def get_prompt_response(chat_fle):
    file_text = lpz.read_markdown_file(chat_file)

    if not lpz.is_smc_content(file_text):
        # stock perplexity files have only a single prompt-response pair
        return [relinker.split_prompt_response_dedup(file_text, split_single_prompt_response_text_perplex)]

    prsplits = []    
    sections = re.split(rf'(?<=\n){lpz.PROMPT_HEADER_SMC}', file_text)
    for section in sections[1:]:  # Process each user section
        section = f'{lpz.PROMPT_HEADER_SMC}\n{section}' # stick header back on for more certtain matching
        dedup_pr = relinker.split_prompt_response_dedup(section, lpz.split_prompt_response_text_smc)
        prsplits.append(dedup_pr)

    return prsplits

verbose = True

num_chat_files = len(chat_files)
all_prompts, all_responses, all_citenums_to_url = [], [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'Parsing {chat_file.stem}')

    for prompt_index, prsplit in enumerate(get_prompt_response(chat_file)):
        print(f'{file_index=}, {prompt_index=}')
        all_prompts.append(prsplit.prompt)
        all_responses.append(prsplit.response_dedup)
        
        citenums_to_url_source = prsplit.citenums_to_url_source.copy()
        citenums_to_url_source[['file_index','chat_file', 'prompt_index']] = file_index, chat_file, prompt_index
        if prsplit.url_to_source_title is not None:
            print('merging titles')
            citenums_to_url_source = citenums_to_url_source.set_index('url')
            citenums_to_url_source['title'] = prsplit.url_to_source_title            
            citenums_to_url_source = citenums_to_url_source.reset_index()
            ic(citenums_to_url_source.columns)
            ic(prsplit.url_to_source_title)
            ic(citenums_to_url_source)
            # citenums_to_url_source = citenums_to_url_source.merge(prsplit.url_to_source_title, on='url', how='outer')
            #citenums_to_url_source = citenums_to_url_source.merge(pd.Series(prsplit.url_to_source_title), on='url', how='outer')
        all_citenums_to_url.append(citenums_to_url_source.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} total citation numbers')

In [ ]:
# Fill in titles when find in other sections (useful for debugging?)
fixed_title_dfs = []
for url, df in all_citenums_to_url.groupby('url'):
    has_no_title = df.title.isna()
    if any(has_no_title):
        if len(titles := df.title[~has_no_title].unique()) > 1:
            ic(url, titles)
            raise ValueError('Different titles for same URL')
        if len(titles) > 0:
            df = df.fillna({'title': titles[0]})
    fixed_title_dfs.append(df)
all_citenums_to_url = pd.concat(fixed_title_dfs)

# fix at the end to allow possible title fill-in from other responses
all_citenums_to_url['title'] = all_citenums_to_url.title.fillna('NO TITLE: likely bare citenum w/ no URL responses')

In [ ]:
all_citenums_to_url

#### Make a unified cite number set for the merged document

In [ ]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Double sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['dedup_num_int'] = df['dedup_num'].astype(int) # so can sort

url_ranks = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_dedup_num_int=('dedup_num_int', 'mean'),
    mean_prompt_index=('prompt_index', 'mean')
).reset_index()

url_ranks = url_ranks.sort_values(by=['mean_file_index', 'mean_prompt_index', 'mean_dedup_num_int'], 
                                 ascending=True).reset_index(drop=True)

# TODO: can I have unif_num as an int?
url_ranks['unif_num'] = np.arange(1, len(url_ranks) + 1).astype(str) # citenum == rank as string

# Merge back the new citenumes

df = df.merge(url_ranks[['url', 'unif_num']], on='url')

In [ ]:
all_citenums_to_url = (df.sort_values(by='unif_num', key=lambda col: col.astype(int))
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('dedup_num_int', axis=1)
                       .set_index('file_index'))

#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [ ]:
all_prompts_same = True
for i in range(0,len(all_prompts)-1):
    is_same = all_prompts[i].strip().lower() == all_prompts[i+1].strip().lower()
    all_prompts_same &= is_same

In [ ]:
# Replace response dedup_num with unif_num, merge prompt/response pairs into single string
is_multi_file_chat = num_chat_files > 1
do_single_top_prompt = all_prompts_same and is_multi_file_chat

full_prompt_indices = rfw.unique_rows(all_citenums_to_url.reset_index(),['file_index','prompt_index'])

# full_ix_names = ['file_index','prompt_index']
# full_prompt_indices = (all_citenums_to_url.reset_index()[full_ix_names]
#                       .sort_values(full_ix_names)
#                       .drop_duplicates()
#                        .reset_index(drop=True))

all_promptresp, chat_source_file_link = '', []
for response_index, (file_index, prompt_index) in full_prompt_indices.iterrows():
    if verbose:
        print(f'Unifying {chat_files[file_index].stem}, {prompt_index=}')
        
    # remap deduped citenums to unified citenums
    citenums_to_url_this = all_citenums_to_url.loc[file_index]
    citenums_dedup_to_unified = citenums_to_url_this.set_index('dedup_num').unif_num.to_dict()
    response_unified = relinker.replace_response_citenums(all_responses[response_index], citenums_dedup_to_unified) # unified citenums

    # Put this body within the appropriate merged dialog headings
    source_link = rfw.file_link_md('source', chat_files[file_index])
    if is_multi_file_chat:
        if do_single_top_prompt:
            if file_index == 0:
                all_promptresp += f'# Prompt\n\n{all_prompts[file_index]}\n# Responses\n'
            all_promptresp += f'\n## {chat_files[file_index].name}\n*{source_link}*\n\n'
            all_promptresp += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)            
        else:
            short_prompt = rfw.get_first_n_words(all_prompts[file_index], lpz.MAX_WORDS_PROMPT_HEADING)
            all_promptresp += f'\n# {short_prompt}\n*{source_link}*\n\n{all_prompts[file_index]}\n## Response\n\n'
            all_promptresp += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)
    else:
        all_promptresp += f'*{source_link}*\n\n# Prompt\n\n{all_prompts[file_index]}\n# Response\n'
        all_promptresp += response_unified

##### Insert links to Obsidian or Zotero

In [ ]:
# unif_num_to_url = all_citenums_to_url[['unif_num', 'url']].drop_duplicates()
# unif_num_to_url.index = unif_num_to_url['unif_num']

# TODO: do better: automate ? handle mixed file types? make all num_dedup plain_links and avoid the problem?
#response_link_type = 'plain_link' # this assumes stock perplexity prompt/resp
response_link_type = 'url_link' # this assumes ALL SMC

# Make the expected function parameter structure
df = rfw.unique_rows(all_citenums_to_url.reset_index(),['url','title'])
url_to_source_title = dict(zip(df.url, df.title))
prsplit_all = lpz.PromptResponseSplitDeDup('','',all_promptresp, all_citenums_to_url, url_to_source_title)

all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit_all, response_link_type)
#all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(all_promptresp_unified , unified_citenums, 'plain_link')
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group('num'))))

print(f'writing to {merged_output_file=}')
merged_output_file.write_text(f'{lpz.make_obsidian_front_matter()}\n{all_promptresp_unified_relinked}\n# Citations\n{relinked_sources}',
                              encoding='utf-8')
print("Done.")